In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Annotated,Literal
from langchain_core.messages import BaseMessage,HumanMessage
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [2]:
load_dotenv()

True

In [3]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage],add_messages]

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

def chat_node(state:ChatState):
    
    #take user query from state
    messages = state["messages"]

    #send to llm
    response = llm.invoke(messages)

    #update state
    return {'messages':[response]}
    

In [5]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

# nodes
graph.add_node("chat_node",chat_node)

# edges
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
thread_id = '1'

while True:

    user_message = input("Type Here :: ")

    print('User :: ', user_message)

    if user_message.strip().lower() in ['exit','bye','quit']:
        break

    config = {'configurable':{'thread_id':thread_id}}

    res = workflow.invoke({"messages":[HumanMessage(content=user_message)]},config=config)

    print('AI response :: ', res['messages'][-1].content)